# 08 · DefiLlama's LI.FI series as a reference snapshot

DefiLlama's daily fees, revenue, supply-side revenue and bridge volume for LI.FI, fetched once without a key into `data/reference/defillama_lifi_daily.csv`, and the adapter that produces them into `data/reference/defillama_lifi_sources_raw.jsonl`. Query `180_defillama_check_against_fee_schedule` reads the snapshot. Every figure here depends on reference data.

The fetch cell reads the snapshot if the file exists and calls the network only when it does not. The raw files are not in this repository, so the committed outputs are the record of the run. DefiLlama can revise past days, so every figure holds as of the fetch time in `data/reference/snapshot_manifest.json`.

Sections: 1 setup · 2 the fetch · 3 what the snapshot holds · 4 the adapter · 5 query 180 against separate checks · 6 date alignment · 7 the last day, the largest day and the months

## 1. Setup

The HTTP helper and the manifest writer follow notebook 01 §1.

In [1]:
import datetime as dt
import hashlib
import json
import time

import pandas as pd
import requests

from lifi import build, paths, run, run_sql

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

REF = paths.reference_dir()
MANIFEST = REF / "snapshot_manifest.json"
DAILY_FILE = "defillama_lifi_daily.csv"
SOURCES_FILE = "defillama_lifi_sources_raw.jsonl"
SIX_CHAINS = ["Ethereum", "Arbitrum", "Base", "OP Mainnet", "Polygon", "BSC"]

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "lifi-takehome-analysis/0.1 (analysis notebook)"


def now_utc():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def get(url, as_json=True, max_tries=5, timeout=120):
    "GET with backoff on HTTP 429, 5xx and network errors. Returns (status code, body)."
    wait = 5.0
    for attempt in range(max_tries):
        try:
            r = SESSION.get(url, timeout=timeout)
        except requests.RequestException as e:
            print(f"  network error, {type(e).__name__}: attempt {attempt + 1}/{max_tries}")
        else:
            if r.status_code != 429 and r.status_code < 500:
                return r.status_code, (r.json() if as_json else r.text)
            print(f"  HTTP {r.status_code}: attempt {attempt + 1}/{max_tries}")
        time.sleep(wait)
        wait *= 2
    raise RuntimeError(f"no response after {max_tries} tries: {url}")


def manifest_write(name, source_url, rows, section, fetched_at, note):
    m = json.loads(MANIFEST.read_text())
    m[name] = {"source_url": source_url, "fetched_at_utc": fetched_at, "rows": int(rows), "notebook_section": section, "note": note}
    MANIFEST.write_text(json.dumps(m, indent=1, sort_keys=True) + "\n")

## 2. The fetch

Eight calls to `api.llama.fi`, no key. `date` is the UTC day of the timestamp DefiLlama gives (§6 tests what that label covers). Inside `totalDataChartBreakdown` each chain holds a value per protocol label; the snapshot sums them per chain, so the parent series `total` keeps the five columns of the other two. Every day and chain the response gives is kept, zeros included. `metric` is the `dataType` asked for; the volume call has none and its adapter returns `dailyBridgeVolume`.

The sources file keeps, per call, everything in the response except the two per-chain structures (`totalDataChartBreakdown`, `chainBreakdown`): the methodology text and `totalDataChart`, against which §3 checks the flattening with no network call. It also keeps the adapter's files at the commit that last touched each of them, so that statements from a web page rest on a fixed copy. They are read as text and never executed.

In [2]:
LLAMA = "https://api.llama.fi/summary"
CALLS = [
    ("total", "dailyFees", f"{LLAMA}/fees/li.fi?dataType=dailyFees"),
    ("total", "dailyRevenue", f"{LLAMA}/fees/li.fi?dataType=dailyRevenue"),
    ("bridge", "dailyFees", f"{LLAMA}/fees/li.fi-bridge-aggregator?dataType=dailyFees"),
    ("bridge", "dailyRevenue", f"{LLAMA}/fees/li.fi-bridge-aggregator?dataType=dailyRevenue"),
    ("bridge", "dailySupplySideRevenue", f"{LLAMA}/fees/li.fi-bridge-aggregator?dataType=dailySupplySideRevenue"),
    ("dex", "dailyFees", f"{LLAMA}/fees/li.fi-dex-aggregator?dataType=dailyFees"),
    ("dex", "dailyRevenue", f"{LLAMA}/fees/li.fi-dex-aggregator?dataType=dailyRevenue"),
    ("bridge", "dailyBridgeVolume", f"{LLAMA}/bridge-aggregators/li.fi-bridge-aggregator"),
]
ADAPTER_REPO = "DefiLlama/dimension-adapters"
ADAPTER_FILES = ["fees/lifi/index.ts", "fees/lifi/feeSources.ts", "fees/lifi-dex-aggregator/index.ts",
                 "bridge-aggregators/lifi/index.ts", "helpers/aggregators/lifi.ts"]


def flatten(series, metric, body):
    rows = []
    for ts, by_chain in body["totalDataChartBreakdown"]:
        assert ts % 86400 == 0, ts
        day = dt.datetime.fromtimestamp(ts, dt.timezone.utc).date().isoformat()
        for chain, by_label in by_chain.items():
            rows.append((day, series, metric, chain, sum(by_label.values())))
    return rows


def fetch_snapshot():
    t0, rows, lines = now_utc(), [], []
    for series, metric, url in CALLS:
        fetched = now_utc()
        status, body = get(url)
        assert status == 200, (url, status)
        rows += flatten(series, metric, body)
        kept = {k: v for k, v in body.items() if k not in ("totalDataChartBreakdown", "chainBreakdown")}
        lines.append({"kind": "defillama_summary", "series": series, "metric": metric, "url": url, "fetched_at_utc": fetched,
                      "http_status": status, "payload": kept})
    for path in ADAPTER_FILES:
        fetched = now_utc()
        commits_url = f"https://api.github.com/repos/{ADAPTER_REPO}/commits?path={path}&sha=master&per_page=1"
        status, commits = get(commits_url)
        assert status == 200 and commits, (commits_url, status)
        sha = commits[0]["sha"]
        lines.append({"kind": "adapter_commit", "path": path, "url": commits_url, "fetched_at_utc": fetched, "http_status": status,
                      "payload": {"sha": sha, "committed_at": commits[0]["commit"]["committer"]["date"], "message": commits[0]["commit"]["message"]}})
        raw_url = f"https://raw.githubusercontent.com/{ADAPTER_REPO}/{sha}/{path}"
        status, text = get(raw_url, as_json=False)
        assert status == 200, (raw_url, status)
        lines.append({"kind": "adapter_file", "path": path, "url": raw_url, "fetched_at_utc": now_utc(), "http_status": status,
                      "payload": {"sha256": hashlib.sha256(text.encode()).hexdigest(), "text": text}})
    daily = pd.DataFrame(rows, columns=["date", "series", "metric", "chain", "value_usd"]).sort_values(["series", "metric", "date", "chain"])
    daily.to_csv(REF / DAILY_FILE, index=False)
    (REF / SOURCES_FILE).write_text("".join(json.dumps(line, sort_keys=True) + "\n" for line in lines))
    t1 = now_utc()
    manifest_write(DAILY_FILE, "; ".join(url for _, _, url in CALLS), len(daily), "08 §2", t0,
                   f"8 calls from {t0} to {t1}, no key; long format from each response's totalDataChartBreakdown; DefiLlama can revise past days, so this date matters")
    manifest_write(SOURCES_FILE, f"the 8 calls of {DAILY_FILE}; https://api.github.com/repos/{ADAPTER_REPO}/commits; https://raw.githubusercontent.com/{ADAPTER_REPO}/<sha>/<path>", len(lines), "08 §2", t0,
                   "one line per call: the responses without the two per-chain structures; for each of the adapter's 5 files the last commit touching it and its text at that commit, read as data and never executed")


if (REF / DAILY_FILE).exists() and (REF / SOURCES_FILE).exists():
    print(f"{DAILY_FILE} and {SOURCES_FILE}: read the snapshots (no network call)")
else:
    print("no snapshot, fetching")
    fetch_snapshot()

daily = pd.read_csv(REF / DAILY_FILE, keep_default_na=False, dtype={"chain": str, "date": str})
sources = [json.loads(line) for line in (REF / SOURCES_FILE).read_text().splitlines()]
entry = json.loads(MANIFEST.read_text())[DAILY_FILE]
print("fetched from:", entry["fetched_at_utc"], "| rows:", entry["rows"], "| source lines:", len(sources))
assert entry["rows"] == len(daily)

defillama_lifi_daily.csv and defillama_lifi_sources_raw.jsonl: read the snapshots (no network call)
fetched from: 2026-11-03T06:49:28+00:00 | rows: 190828 | source lines: 18


## 3. What the snapshot holds

One row per series and metric. Then the flattening's check: for each call, the sum of the chains on each day against that response's own `totalDataChart`, with no network call. `first_nonzero` shows when each series starts carrying values.

In [3]:
summary = (daily.groupby(["series", "metric"])
           .agg(rows=("value_usd", "size"), first_date=("date", "min"), last_date=("date", "max"), chains=("chain", "nunique"), days=("date", "nunique"), sum_usd=("value_usd", "sum"))
           .reset_index())
first_nonzero = daily[daily["value_usd"] != 0].groupby(["series", "metric"])["date"].min().rename("first_nonzero").reset_index()
summary.merge(first_nonzero, on=["series", "metric"])

,series,metric,rows,first_date,last_date,chains,days,sum_usd,first_nonzero
0,bridge,dailyBridgeVolume,36650,2024-12-31,2026-11-03,63,671,"11,090,161,982.25",2024-12-31
1,bridge,dailyFees,30324,2024-08-01,2026-11-03,45,825,"29,368,175.01",2024-08-01
2,bridge,dailyRevenue,30324,2024-08-01,2026-11-03,45,825,"4,440,328.19",2024-08-01
3,bridge,dailySupplySideRevenue,30324,2024-08-01,2026-11-03,45,825,"24,926,251.76",2024-08-01
4,dex,dailyFees,1279,2026-09-30,2026-11-03,37,35,"836,999.92",2026-09-30
5,dex,dailyRevenue,1279,2026-09-30,2026-11-03,37,35,"263,864.21",2026-09-30
6,total,dailyFees,30324,2024-08-01,2026-11-03,45,825,"30,205,174.93",2024-08-01
7,total,dailyRevenue,30324,2024-08-01,2026-11-03,45,825,"4,704,192.40",2024-08-01


In [4]:
checks = []
for line in sources:
    if line["kind"] != "defillama_summary":
        continue
    chart = pd.DataFrame(line["payload"]["totalDataChart"], columns=["ts", "chart"])
    chart["date"] = pd.to_datetime(chart["ts"], unit="s").dt.strftime("%Y-%m-%d")
    part = daily[(daily["series"] == line["series"]) & (daily["metric"] == line["metric"])].groupby("date")["value_usd"].sum().rename("chains").reset_index()
    both = chart.merge(part, on="date", how="outer")
    checks.append({"series": line["series"], "metric": line["metric"], "days_chart": len(chart), "days_chains": len(part), "days_unmatched": int(both[["chart", "chains"]].isna().any(axis=1).sum()),
                   "sum_chart": both["chart"].sum(), "sum_chains": both["chains"].sum(), "largest_day_gap": (both["chart"] - both["chains"]).abs().max()})
pd.DataFrame(checks)

,series,metric,days_chart,days_chains,days_unmatched,sum_chart,sum_chains,largest_day_gap
0,total,dailyFees,825,825,0,"30,205,177.17","30,205,174.93",2.35
1,total,dailyRevenue,825,825,0,"4,704,194.15","4,704,192.40",1.76
2,bridge,dailyFees,825,825,0,"29,368,175.17","29,368,175.01",2.35
3,bridge,dailyRevenue,825,825,0,"4,440,328.15","4,440,328.19",1.76
4,bridge,dailySupplySideRevenue,825,825,0,"24,926,253.02","24,926,251.76",1.97
5,dex,dailyFees,35,35,0,"837,002.00","836,999.92",0.47
6,dex,dailyRevenue,35,35,0,"263,866.00","263,864.21",0.50
7,bridge,dailyBridgeVolume,671,671,0,"11,090,161,995.00","11,090,161,982.25",2.97


`total` against `bridge` plus `dex`, per day and metric: the largest gap is rounding. Before 2026-09-30 the `dex` series has no rows, so `bridge` equals `total` there and holds same-chain swaps too. That the sum holds on 2026-09-30 does not make that day clean: §7 shows `bridge` and `dex` identical on it.

In [5]:
wide = (daily[daily["metric"].isin(["dailyFees", "dailyRevenue"])].groupby(["date", "metric", "series"])["value_usd"].sum().unstack("series").fillna(0))
wide["gap"] = wide["total"] - wide["bridge"] - wide["dex"]
split = wide.index.get_level_values("date") >= "2026-09-30"
pd.DataFrame({"largest_gap_on_or_after_2026_09_30": [wide.loc[split, "gap"].abs().max()], "largest_gap_before": [wide.loc[~split, "gap"].abs().max()], "dex_before": [wide.loc[~split, "dex"].sum()]})

,largest_gap_on_or_after_2026_09_30,largest_gap_before,dex_before
0,0.00,0.00,0.00


## 4. The adapter

The commits below are the last to touch each file on `master` at fetch time. The methodology text is DefiLlama's own, from the responses. The lines that follow come from the committed source text and are what `data/reference/README.md` state.

In [6]:
pd.DataFrame([{"path": l["path"], "commit": l["payload"]["sha"], "committed_at": l["payload"]["committed_at"], "message": l["payload"]["message"].splitlines()[0]}
              for l in sources if l["kind"] == "adapter_commit"])

,path,commit,committed_at,message
0,fees/lifi/index.ts,5a01404eb9ae19297ef895794af97519e46e5811,2026-10-19T06:49:41Z,use token labels
1,fees/lifi/feeSources.ts,a3d2152ee68f1f941c2e59e6f282f35a511116ed,2026-10-09T15:17:43Z,"track jumper fees on solana, bitcoin, sui, hyperevm and api chains..."
2,fees/lifi-dex-aggregator/index.ts,0f8575b29b7efe3c26f32b26b08727aba3bee311,2026-09-30T20:03:46Z,Split LI.FI and Jumper bridge and DEX fee series (#9809)
3,bridge-aggregators/lifi/index.ts,d55e834fcfcd5faa21d8045c80d9e99c4bd3420c,2026-09-25T18:50:02Z,lifi: include Jumper volume (remove integrator exclusion) (#8972)
4,helpers/aggregators/lifi.ts,8a0f254f631125f49d77c233116b1cb5c940fcb4,2026-10-18T16:20:34Z,smol fixes (#10263)


In [7]:
for line in sources:
    if line["kind"] == "defillama_summary" and line["metric"] in ("dailyFees", "dailyBridgeVolume"):
        print(line["series"], line["metric"], json.dumps(line["payload"].get("methodology"), indent=1))

total dailyFees null
bridge dailyFees {
 "Fees": "All fees paid by users on LI.FI bridges, including integrator fees.",
 "ProtocolRevenue": "Share of fees kept by LI.FI.",
 "Revenue": "Share of fees kept by LI.FI.",
 "SupplySideRevenue": "Share of fees paid to integrators, including Jumper since 2026-08-07."
}
dex dailyFees {
 "Fees": "All fees paid by users on LI.FI swaps, including integrator fees.",
 "ProtocolRevenue": "Share of fees kept by LI.FI.",
 "Revenue": "Share of fees kept by LI.FI.",
 "SupplySideRevenue": "Share of fees paid to integrators, including Jumper since 2026-08-07."
}
bridge dailyBridgeVolume {}


In [8]:
texts = {l["path"]: l["payload"]["text"] for l in sources if l["kind"] == "adapter_file"}


def show(path, *needles):
    for i, text in enumerate(texts[path].splitlines(), start=1):
        if any(n in text for n in needles):
            print(f"{path}:{i}: {text.strip()[:170]}")


show("fees/lifi/index.ts", "getDefaultDexTokensBlacklisted(options", "blacklistForChain.has", "OtherFee;", "separateJumper =", "Jumper is a separate", "no LI.FI cut", "String(fee.recipient)", "copy per adapter", "lifi-dex-aggregator run the bridge fetch")
show("fees/lifi/feeSources.ts", "first Jumper fee", "and swaps 2026", "JumperFeeStart =", "JumperIntegrators =", "The recipient in every", "transaction.kind = 'bridge'", "event FeesCollected", "event FeesForwarded")
show("bridge-aggregators/lifi/index.ts", "LIFI_API_CHAINS.includes", "count all integrators", "dailyVolume.add")
show("helpers/aggregators/lifi.ts", "CHAIN.BITCOIN, CHAIN.SOLANA", "CHAIN.TAIKO, CHAIN.FLARE", "CHAIN.MOONBEAM, CHAIN.FRAXTAL", "MAX_TRANSFER_USD =", "tx.status === 'DONE' &&")
print()
print([l for l in sources if l["kind"] == "adapter_commit" and l["path"] == "fees/lifi-dex-aggregator/index.ts"][0]["payload"]["message"].split("Co-authored")[0])

fees/lifi/index.ts:19: const blacklistForChain = new Set(getDefaultDexTokensBlacklisted(options.chain));
fees/lifi/index.ts:22: if (blacklistForChain.has(token.toLowerCase())) return;
fees/lifi/index.ts:42: if (!kind && category === 'bridge') return OtherFee;
fees/lifi/index.ts:52: const separateJumper = options.startTimestamp >= Date.parse(JumperFeeStart) / 1000;
fees/lifi/index.ts:55: // Jumper is a separate integrator whose fee is paid into LI.FI's wallet as a single leg (2/5 bps,
fees/lifi/index.ts:56: // no LI.FI cut): count it as integrator fee, not LI.FI revenue.
fees/lifi/index.ts:61: addFee(log.args.token, fee.amount, !jumper && String(fee.recipient).toLowerCase() === LifiRecipient, source);
fees/lifi/index.ts:77: // copy per adapter: the runner writes fetch onto each chain entry, so a shared object makes
fees/lifi/index.ts:78: // lifi-dex-aggregator run the bridge fetch (or vice versa)
fees/lifi/feeSources.ts:5: export const FeeCollectedEvent = "event FeesCollected(address in

## 5. Query 180 against separate checks

`run("180_defillama_check_against_fee_schedule")`. `source` says where a check comes from: a one-off fetch of the same day, or a recomputation from the CSV. This table covers only the figures it lists, not every row of the query. `over_1pct` flags a figure more than 1% from its check. The build runs only if the database predates the snapshot.

In [9]:
try:
    run_sql("SELECT 1 FROM ref_defillama_lifi_daily LIMIT 1")
except Exception:
    build(out=None)

q180 = run("180_defillama_check_against_fee_schedule")
q180[q180["section"].str[0].isin(["1", "2"])].dropna(axis=1, how="all")

,section,item,measure,days,last_day,defillama_usd,ours_usd,difference_usd,difference_pct,annual_usd,share_pct
0,1. Same scope,"volume, started",volume,7,2026-10-19,"306,113,735.00","307,969,358.59","-1,855,623.59",-0.60,NaN,NaN
1,1. Same scope,"volume, bridge volume",volume,7,2026-10-19,"306,113,735.00","307,741,849.38","-1,628,114.38",-0.53,NaN,NaN
2,1. Same scope,all fees,fees,7,2026-10-19,"134,463.00","135,051.74",-588.74,-0.44,NaN,NaN
3,1. Same scope,LI.FI fee without Jumper,revenue,7,2026-10-19,"24,883.00","25,282.97",-399.97,-1.58,NaN,NaN
4,1. Same scope,integrators' fees with Jumper,supply-side revenue,7,2026-10-19,"109,580.00","109,768.77",-188.77,-0.17,NaN,NaN
5,2. Scope ladder,"bridges, six chains",revenue,7,2026-10-19,"24,883.00",NaN,NaN,NaN,"1,293,916.00",NaN
6,2. Scope ladder,"bridges, six chains",fees,7,2026-10-19,"134,463.00",NaN,NaN,NaN,"6,992,076.00",NaN
7,2. Scope ladder,"bridges, all chains",revenue,7,2026-10-19,"28,822.49",NaN,NaN,NaN,"1,498,769.00",NaN
8,2. Scope ladder,"bridges, all chains",fees,7,2026-10-19,"157,767.63",NaN,NaN,NaN,"8,203,917.00",NaN
9,2. Scope ladder,"bridges plus same-chain swaps, all chains",revenue,7,2026-10-19,"51,025.44",NaN,NaN,NaN,"2,653,323.00",NaN


In [10]:
CHECKS = [
    ("1. Same scope", "volume, started", "defillama_usd", 306.1e6), ("1. Same scope", "volume, started", "ours_usd", 307.97e6),
    ("1. Same scope", "all fees", "defillama_usd", 134463), ("1. Same scope", "all fees", "ours_usd", 135052),
    ("1. Same scope", "LI.FI fee without Jumper", "defillama_usd", 24883), ("1. Same scope", "LI.FI fee without Jumper", "ours_usd", 25283),
    ("1. Same scope", "integrators' fees with Jumper", "defillama_usd", 109580), ("1. Same scope", "integrators' fees with Jumper", "ours_usd", 109769),
    ("2. Scope ladder", "bridges, all chains", "revenue", 28822), ("2. Scope ladder", "bridges, all chains", "fees", 157768),
    ("2. Scope ladder", "bridges plus same-chain swaps, all chains", "revenue", 51026), ("2. Scope ladder", "bridges plus same-chain swaps, all chains", "fees", 283177),
    ("2. Scope ladder", "six chains' share of bridges on all chains", "bridge volume", 78.0), ("2. Scope ladder", "six chains' share of bridges on all chains", "LI.FI fee", 86.0),
    ("3. Calendar years", "2025", "fees", 15265751), ("3. Calendar years", "2025", "revenue", 2252921),
    ("3. Calendar years", "2026 to the snapshot date", "fees", 11263241), ("3. Calendar years", "2026 to the snapshot date", "revenue", 2122252),
    ("3. Calendar years", "2026 annualised by days", "fees", 13.4e6), ("3. Calendar years", "2026 annualised by days", "revenue", 2.5e6),
]
REVIEW_CHECKS = [
    ("3. Calendar years", "2026 annualised by days, without the largest revenue day", "fees", 13221260.96),
    ("3. Calendar years", "2026 annualised by days, without the largest revenue day", "revenue", 2369843.48),
    ("3. Calendar years", "2026 annualised by days, with 2026-09-30 counted once", "fees", 13343400),
    ("3. Calendar years", "2026 annualised by days, with 2026-09-30 counted once", "revenue", 2515729),
    ("3. Calendar years", "2026 annualised by days, without the largest revenue day, with 2026-09-30 counted once", "revenue", 2362348),
    ("3. Calendar years", "2026-09-30, bridge series", "fees", 40161.72), ("3. Calendar years", "2026-09-30, bridge series", "revenue", 6283.71),
    ("3. Calendar years", "2026-09-30, dex series", "fees", 40161.72), ("3. Calendar years", "2026-09-30, dex series", "revenue", 6283.71),
    ("3. Calendar years", "2026-09-30, total series", "fees", 80323.44), ("3. Calendar years", "2026-09-30, total series", "revenue", 12567.42),
    ("3. Calendar years", "bridge series, mean a day, 2026-09-17 to 2026-09-29", "fees", 77844),
    ("3. Calendar years", "bridge series, mean a day, 2026-09-30 to 2026-10-13", "fees", 29161),
]
rows = []
for source, section, item, what, check in [("one-off fetch", *c) for c in CHECKS] + [("recomputation", *c) for c in REVIEW_CHECKS]:
    if what in ("defillama_usd", "ours_usd"):
        figure = q180[(q180["section"] == section) & (q180["item"] == item)][what].iloc[0]
    else:
        hit = q180[(q180["section"] == section) & (q180["item"] == item) & (q180["measure"] == what)].iloc[0]
        figure = hit["share_pct"] if "share" in item else hit["defillama_usd"]
    rows.append({"source": source, "section": section, "item": item, "what": what, "figure": figure, "check": check, "pct_from_check": 100 * (figure - check) / check, "over_1pct": abs(figure - check) / check > 0.01})
pd.DataFrame(rows)

,source,section,item,what,figure,check,pct_from_check,over_1pct
0,one-off fetch,1. Same scope,"volume, started",defillama_usd,"306,113,735.00","306,100,000.00",0.00,False
1,one-off fetch,1. Same scope,"volume, started",ours_usd,"307,969,358.59","307,970,000.00",-0.00,False
2,one-off fetch,1. Same scope,all fees,defillama_usd,"134,463.00","134,463.00",0.00,False
3,one-off fetch,1. Same scope,all fees,ours_usd,"135,051.74","135,052.00",-0.00,False
4,one-off fetch,1. Same scope,LI.FI fee without Jumper,defillama_usd,"24,883.00","24,883.00",0.00,False
5,one-off fetch,1. Same scope,LI.FI fee without Jumper,ours_usd,"25,282.97","25,283.00",-0.00,False
6,one-off fetch,1. Same scope,integrators' fees with Jumper,defillama_usd,"109,580.00","109,580.00",0.00,False
7,one-off fetch,1. Same scope,integrators' fees with Jumper,ours_usd,"109,768.77","109,769.00",-0.00,False
8,one-off fetch,2. Scope ladder,"bridges, all chains",revenue,"28,822.49","28,822.00",0.00,False
9,one-off fetch,2. Scope ladder,"bridges, all chains",fees,"157,767.63","157,768.00",-0.00,False


## 6. Date alignment

The snapshot's last label is the day of the fetch, so a label may not be the UTC day it names. If a label covered the day before, the week 13 to 19 October in query 180 would read 12 to 18 October. Query 180's section 4 tests it: DefiLlama's bridge volume on the six chains is built from the same `LiFiTransferStarted` events as the file (§4), so over the 42 chain-days it should track started volume only at the right shift. The fee and revenue rows do the same with the schedule's rows on the day of their own event; the schedule's weights are per string, so those rows are noisier and test the direction only. The next table shows the same section by day and chain.

In [11]:
section4 = q180[q180["section"] == "4. By day and chain"]
section4[section4["item"].str.startswith("label")][["item", "measure", "defillama_usd", "ours_usd", "chain_day_correlation", "chain_day_abs_gap_usd"]]

,item,measure,defillama_usd,ours_usd,chain_day_correlation,chain_day_abs_gap_usd
109,label = event day - 1,fees,"137,704.00","135,051.74",0.87,"49,439.00"
110,label = event day - 1,revenue,"24,957.00","25,282.97",0.88,"9,249.00"
111,label = event day - 1,volume,"296,836,416.00","307,969,358.59",0.64,"158,288,407.00"
112,label = event day,fees,"134,463.00","135,051.74",0.88,"43,529.00"
113,label = event day,revenue,"24,883.00","25,282.97",0.94,"6,343.00"
114,label = event day,volume,"306,113,735.00","307,969,358.59",1.00,"3,088,997.00"
115,label = event day + 1,fees,"132,925.00","135,051.74",0.86,"55,070.00"
116,label = event day + 1,revenue,"24,677.00","25,282.97",0.79,"12,553.00"
117,label = event day + 1,volume,"289,954,850.00","307,969,358.59",0.63,"161,348,414.00"


In [12]:
parts = section4[~section4["item"].str.startswith("label")]
parts.pivot(index="item", columns="measure", values="difference_pct")[["volume", "fees", "revenue"]]

measure,volume,fees,revenue
item,,,
2026-10-13,0.30,10.60,-6.07
2026-10-14,-5.03,9.85,-6.68
2026-10-15,-0.05,31.89,13.70
2026-10-16,0.05,22.36,14.64
2026-10-17,-0.14,-23.23,-30.60
2026-10-18,0.01,-4.92,4.31
2026-10-19,-0.06,-31.32,5.59
Arbitrum,0.38,2.20,-19.20
BSC,0.49,11.17,16.18


## 7. The last day, the largest day and the months

Query 180's section 3. The fetch ran at about 06:49 UTC on the snapshot's last label, and that label already holds a value the size of a whole day, so it is not a completed UTC day. The annualised rows show what that day moves, and what the largest revenue day moves. The months show how far 2026 is from a steady rate.

The rows after the months bear on two limits. On 2026-09-30 `bridge` and `dex` are identical on every chain and `total` sums them, so the "counted once" rows take one of the two out. The mean rows are a day's mean, and the step in them, with the commit printed in §4, is why the stored days 2026-08-07 to 2026-09-29 may still hold Jumper's fee in revenue. In the `days` column, the row "days from 2026-09-30 with bridge equal to dex" counts such days and its `last_day` names the latest.

In [13]:
section3 = q180[q180["section"] == "3. Calendar years"]
section3.pivot(index="item", columns="measure", values="defillama_usd").reindex(section3["item"].drop_duplicates())

measure,fees,revenue
item,,
2025,"15,265,749.72","2,252,918.42"
2026 to the snapshot date,"11,263,240.67","2,122,252.91"
2026 annualised by days,"13,391,149.33","2,523,199.71"
"2026 annualised by days, without the largest revenue day","13,221,260.96","2,369,843.48"
"2026 annualised by days, without the snapshot's last day","13,396,829.34","2,522,965.55"
largest revenue day of 2026,"179,115.04","135,480.02"
the snapshot's last day,"31,926.21","7,109.19"
"largest revenue day of 2026, dex series, Ethereum","129,343.00","125,877.00"
2026-01,"498,896.00","170,916.00"


In [14]:
section3[section3["item"].str.startswith("days from")][["item", "measure", "days", "last_day"]]

,item,measure,days,last_day
58,days from 2026-09-30 with bridge equal to dex on every chain,fees,1,2026-09-30
59,days from 2026-09-30 with bridge equal to dex on every chain,revenue,1,2026-09-30
